# Chapter 7 — Part II

In [1]:
import chainladder as cl
from IPython.display import display, HTML, Markdown
import numpy as np
import pandas as pd


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

import warnings

warnings.filterwarnings(
    "ignore",
    message=r"Some exclusions have been ignored\..*link ratio\(s\) is required.*",
    category=UserWarning,
)


#pd.options.display.float_format = '{:,.0f}'.format

## Exhibit III — U.S. Industry P.P. Auto (Impact of Changing Conditions)

In order to illustrate the assumptions and corresponding limitations of the Development method, Exhibit III applies the method to the **U.S. Private Passenger Industry Auto** (USPP Auto) example under four different scenarios:

1. stable claim ratios and no change in case outstanding strength (steady state);
1. increasing claim ratios but no change in case outstanding strength;
1. stable claim ratios but increasing case outstanding strength; and
1. increasing claim ratios and increasing case outstanding strength.

The key assumptions of this study are as follows:

- the earned premium for the first year (1999) is assumed to be $1 million with a 5% annual premium trend;
- (discuss assumed claim ratios)

Since the USPP Auto dataset is contained in the `chainladder` package, we simply load it into a triangle as follows:

In [2]:
triangles = cl.load_sample("friedland_uspp")

In [3]:
def dev_exhibit(tri: cl.Triangle, avg_params: dict[str,int], selected_avg: str, tail: float) -> dict[cl.Triangle()]:
    display('')
    display(HTML("""
    <h2 style='text-align:left;'>
    PART 1 - Data Triangle
    </h2>
    """))
    display(tri)
    display(HTML("""
    <h2 style='text-align:left;'>
    PART 2 - Age-to-Age Factors
    </h2>
    """))
    age_to_age_df = tri.age_to_age.to_frame(origin_as_datetime=False)
    display(
        age_to_age_df.style.format(precision=3, na_rep="")
    )
    devs = {}
    display(HTML("""
    <h2 style='text-align:left;'>
    PART 3 - Average Age-to-Age Factor
    </h2>
    """))
    for k,v in avg_params.items():
        devs[k] = cl.Development(**v).fit_transform(tri)
    def print_ldfs(ldf_dict:dict[cl.Triangle()]):
        with pd.option_context("display.float_format", "{:.3f}".format):
            display(pd.concat([v.to_frame().rename(index={'(All)':k}) for k,v in ldf_dict.items()]))
        return None
    print_ldfs({k:v.ldf_.round(decimals=3) for k,v in devs.items()})
    devs["Selected"] = cl.TailConstant(tail = tail, projection_period = 0).fit_transform(devs[selected_avg])
    selected = {}
    selected['CDF to Ultimate'] = devs["Selected"].ldf_.round(decimals=3).incr_to_cum().round(decimals=3)
    selected['Percent Reported'] = (1/selected['CDF to Ultimate']).round(decimals=3)
    display(HTML("""
    <h2 style='text-align:left;'>
    PART 4 - Selected Age-to-Age Factor
    </h2>
    """))
    print_ldfs({'Selected':devs['Selected'].ldf_.round(decimals=3)})
    print_ldfs(selected)
    return devs

In [4]:
#| hide-input

def ex3sht1(
          tri: cl.Triangle,
          dev_input: dict,
          tail_input: dict
) -> tuple:
    
    def format_col(x, reverse = False):
            if reverse is True:
                return x.to_frame().squeeze()[::-1].reset_index(drop=True)
            return x.to_frame().squeeze().reset_index(drop=True)

    def col_diff(x, y):
        return x.squeeze() - y.squeeze()

    def col_div(x, y):
        return x.squeeze() / y.squeeze()

    def col_mult(x, y):
        return x.squeeze() * y.squeeze()

    title = ("Exhibit III, Sheet 1, Summary of Earned Premium and Claim Ratio Assumptions and Actual IBNR")
    #scenario = "Steady State"

    dev = cl.Development(**dev_input).fit(tri)
    model = cl.Chainladder().fit(dev.transform(tri))
    tail = cl.TailConstant(**tail_input).fit_transform(dev.transform(tri))
    ult = model.ultimate_
    ibnr = model.ibnr_

    col1 = pd.DataFrame({"Accident Year" : format_col(tri.origin.astype(str))})
    col2 = pd.DataFrame({"Earned Premium" : format_col(tri.latest_diagonal.loc["Steady State", "Earned Premium"])})
    col4 = pd.DataFrame({"Ult. Claims" : format_col(ult.loc["Steady State", "Reported Claims"])})
    col3 = pd.DataFrame({"Ult. Claim Ratio" : col_div(col4, col2)})
    col5 = pd.DataFrame({"Rep. Claims 12/31/08" : format_col(tri.latest_diagonal.loc["Steady State", "Reported Claims"])})
    col6 = pd.DataFrame({"Actual IBNR" : col_diff(col4, col5)})
    col8 = pd.DataFrame({"Ult. Claims" : format_col(ult.loc["Increasing Claim", "Reported Claims"])})
    col9 = pd.DataFrame({"Rep. Claims 12/31/08" : format_col(tri.latest_diagonal.loc["Increasing Claim", "Reported Claims"])})
    col7 = pd.DataFrame({"Ult. Claim Ratio" : col_div(col8, col2)})
    col10 = pd.DataFrame({"Actual IBNR" : col_diff(col8, col9)})
    col11 = pd.DataFrame({"Accident Year" : format_col(tri.origin.astype(str))})
    col12 = pd.DataFrame({"Earned Premium" : format_col(tri.latest_diagonal.loc["Steady State", "Earned Premium"])})
    col14 = pd.DataFrame({"Ult. Claims" : format_col(ult.loc["Steady State", "Reported Claims"])})
    col13 = pd.DataFrame({"Ult. Claim Ratio" : col_div(col14, col12)})
    col15 = pd.DataFrame({"Rep. Claims 12/31/08" : format_col(tri.latest_diagonal.loc["Increasing Case", "Reported Claims"])})
    col16 = pd.DataFrame({"Actual IBNR" : col_diff(col14, col15)})
    col18 = pd.DataFrame({"Ult. Claims" : format_col(ult.loc["Increasing Claim", "Reported Claims"])})
    col17 = pd.DataFrame({"Ult. Claim Ratio" : col_div(col18, col12)})
    col19 = pd.DataFrame({"Rep. Claims 12/31/08" : format_col(tri.latest_diagonal.loc["Increasing Claim Case", "Reported Claims"])})
    col20 = pd.DataFrame({"Actual IBNR" : col_diff(col18, col19)})

    cols = [col1, col2, col3, col4, col5, col6]
    df1 = pd.concat([col1, col2], axis=1)
    df2 = pd.concat([col3, col4, col5, col6], axis=1)
    df3 = pd.concat([col7, col8, col9, col10], axis=1)
    df4 = pd.concat([col11, col12], axis=1)
    df5 = pd.concat([col13, col14, col15, col16], axis=1)
    df6 = pd.concat([col17, col18, col19, col20], axis=1)

    dfs_upper = [df1, df2, df3]
    dfs_lower = [df4, df5, df6]

    results_upper = pd.concat(dfs_upper, axis=1, keys=["", "Steady State", "Increasing Claim Ratios"])
    results_upper = results_upper.set_index(("", "Accident Year")) #since its multi-index, I needed to use the tuple to designate the index col
    results_upper.index.name = "Accident Year" 
    results_upper.loc["Total"] = results_upper.sum()

    #remove the totals from the ratio columns (where totals do not make sense)
    results_upper.loc["Total", ("Steady State", "Ult. Claim Ratio")] = np.nan
    results_upper.loc["Total", ("Increasing Claim Ratios", "Ult. Claim Ratio")] = np.nan

    results_lower = pd.concat(dfs_lower, axis=1, keys=["", "Increasing Case Outstanding Strength", "Increasing Claim Ratios and Case Outstanding Strength"])
    results_lower = results_lower.set_index(("", "Accident Year")) #since its multi-index, I needed to use the tuple to designate the index col
    results_lower.index.name = "Accident Year" 
    results_lower.loc["Total"] = results_lower.sum()

    #remove the totals from the ratio columns (where totals do not make sense)
    results_lower.loc["Total", ("Increasing Case Outstanding Strength", "Ult. Claim Ratio")] = np.nan
    results_lower.loc["Total", ("Increasing Claim Ratios and Case Outstanding Strength", "Ult. Claim Ratio")] = np.nan

    display(HTML("""
    <h2 style='text-align:center;'>
    Exhibit III Sheet 1: Summary of Earned Premium and Claim Ratio Assumptions and Actual IBNR
    </h2>
    """))
    
    upper_formats = {
        ("Steady State", "Ult. Claim Ratio"): "{:.1%}",
        ("Steady State", "Ult. Claims"): "{:,.0f}",
        ("Steady State", "Rep. Claims 12/31/08"): "{:,.0f}",
        ("Steady State", "Actual IBNR"): "{:,.0f}",

        ("Increasing Claim Ratios", "Ult. Claim Ratio"): "{:.1%}",
        ("Increasing Claim Ratios", "Ult. Claims"): "{:,.0f}",
        ("Increasing Claim Ratios", "Rep. Claims 12/31/08"): "{:,.0f}",
        ("Increasing Claim Ratios", "Actual IBNR"): "{:,.0f}",

        ("", "Earned Premium"): "{:,.0f}",
    }

    display(
        results_upper.style.format(
             upper_formats,
             na_rep=""
        )
    )
    
    lower_formats = {
        ("Increasing Case Outstanding Strength", "Ult. Claim Ratio"): "{:.1%}",
        ("Increasing Case Outstanding Strength", "Ult. Claims"): "{:,.0f}",
        ("Increasing Case Outstanding Strength", "Rep. Claims 12/31/08"): "{:,.0f}",
        ("Increasing Case Outstanding Strength", "Actual IBNR"): "{:,.0f}",

        (
            "Increasing Claim Ratios and Case Outstanding Strength",
            "Ult. Claim Ratio",
        ): "{:.1%}",
        (
            "Increasing Claim Ratios and Case Outstanding Strength",
            "Ult. Claims",
        ): "{:,.0f}",
        (
            "Increasing Claim Ratios and Case Outstanding Strength",
            "Rep. Claims 12/31/08",
        ): "{:,.0f}",
        (
            "Increasing Claim Ratios and Case Outstanding Strength",
            "Actual IBNR",
        ): "{:,.0f}",

        ("", "Earned Premium"): "{:,.0f}",
    }

    display(
         results_lower.style.format(
              lower_formats,
              na_rep=""
            )
    )

    return (results_upper, results_lower)

In [5]:
dev_input = {
    "average" : "volume",
    "n_periods" : 5
}

tail_input = {
    "tail" : 1.0,
    "projection_period" : -1
}
res_up, res_low = ex3sht1(triangles, dev_input, tail_input)

### Exhibit III, Sheet 2 - Sheet 2: USPP Auto Steady-State - Reported Claims

We then generate Sheet 2, Parts 1-4 which contain the data triangle and age-to-age factors for USPP reported claims under the steady-state scenario (p. 115).

In [6]:
_ = dev_exhibit(triangles.loc["Steady State", "Reported Claims"],avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},selected_avg='volume_5',tail=1)

''

,12,24,36,48,60,72,84,96,108,120
1999,"539,000","630,000","665,000","686,000","693,000","693,000","700,000","700,000","700,000","700,000"
2000,"565,950","661,500","698,250","720,300","727,650","727,650","735,000","735,000","735,000",
2001,"594,248","694,575","733,163","756,315","764,033","764,033","771,750","771,750",,
2002,"623,960","729,304","769,821","794,131","802,234","802,234","810,338",,,
2003,"655,158","765,769","808,312","833,837","842,346","842,346",,,,
2004,"687,916","804,057","848,727","875,529","884,463",,,,,
2005,"722,312","844,260","891,164","919,306",,,,,,
2006,"758,427","886,473","935,722",,,,,,,
2007,"796,348","930,797",,,,,,,,
2008,"836,166",,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1999,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,1.000
2000,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,
2001,1.169,1.056,1.032,1.010,1.000,1.010,1.000,,
2002,1.169,1.056,1.032,1.010,1.000,1.010,,,
2003,1.169,1.056,1.032,1.010,1.000,,,,
2004,1.169,1.056,1.032,1.010,,,,,
2005,1.169,1.056,1.032,,,,,,
2006,1.169,1.056,,,,,,,
2007,1.169,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
volume_5,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,1.000


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120,120-132
Selected,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,1.000,1.000


,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult,120-Ult
CDF to Ultimate,1.300,1.112,1.053,1.020,1.010,1.010,1.000,1.000,1.000,1.000
Percent Reported,0.769,0.899,0.950,0.980,0.990,0.990,1.000,1.000,1.000,1.000


### Exhibit III, Sheet 3: USPP Auto Steady-State - Paid Claims

We then generate Sheet 3, Parts 1-4 which contain the data triangle and age-to-age factors for USPP paid claims under the steady-state scenario (p. 116).

In [7]:
_ = dev_exhibit(triangles.loc["Steady State", "Paid Claims"],avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},selected_avg='volume_5',tail=1)

''

,12,24,36,48,60,72,84,96,108,120
1999,"294,000","497,000","588,000","644,000","672,000","686,000","693,000","693,000","700,000","700,000"
2000,"308,700","521,850","617,400","676,200","705,600","720,300","727,650","727,650","735,000",
2001,"324,135","547,943","648,270","710,010","740,880","756,315","764,033","764,033",,
2002,"340,342","575,340","680,684","745,511","777,924","794,131","802,234",,,
2003,"357,359","604,107","714,718","782,786","816,820","833,837",,,,
2004,"375,227","634,312","750,454","821,925","857,661",,,,,
2005,"393,988","666,028","787,976","863,022",,,,,,
2006,"413,688","699,329","827,375",,,,,,,
2007,"434,372","734,295",,,,,,,,
2008,"456,090",,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1999,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000
2000,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,
2001,1.690,1.183,1.095,1.043,1.021,1.010,1.000,,
2002,1.690,1.183,1.095,1.043,1.021,1.010,,,
2003,1.690,1.183,1.095,1.043,1.021,,,,
2004,1.690,1.183,1.095,1.043,,,,,
2005,1.690,1.183,1.095,,,,,,
2006,1.690,1.183,,,,,,,
2007,1.690,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
volume_5,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120,120-132
Selected,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000,1.000


,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult,120-Ult
CDF to Ultimate,2.378,1.407,1.190,1.086,1.042,1.020,1.010,1.010,1.000,1.000
Percent Reported,0.421,0.711,0.840,0.921,0.960,0.980,0.990,0.990,1.000,1.000


### Exhibit III, Sheet 4: USPP Auto Increasing Claim Ratios - Reported Claims

We then generate Sheet 4, Parts 1-4 which contain the data triangle and age-to-age factors for USPP reported claims under the increasing claim ratio scenario (p. 117).

In [37]:
_ = dev_exhibit(triangles.loc["Increasing Claim", "Reported Claims"],avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},selected_avg='volume_5',tail=1)

''

,12,24,36,48,60,72,84,96,108,120
1999,"539,000","630,000","665,000","686,000","693,000","693,000","700,000","700,000","700,000","700,000"
2000,"565,950","661,500","698,250","720,300","727,650","727,650","735,000","735,000","735,000",
2001,"594,248","694,575","733,163","756,315","764,033","764,033","771,750","771,750",,
2002,"623,960","729,304","769,821","794,131","802,234","802,234","810,338",,,
2003,"655,158","765,769","808,312","833,837","842,346","842,346",,,,
2004,"786,189","918,923","969,974","1,000,605","1,010,815",,,,,
2005,"877,093","1,025,173","1,082,127","1,116,300",,,,,,
2006,"975,121","1,139,751","1,203,071",,,,,,,
2007,"1,080,759","1,263,224",,,,,,,,
2008,"1,194,523",,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1999,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,1.000
2000,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,
2001,1.169,1.056,1.032,1.010,1.000,1.010,1.000,,
2002,1.169,1.056,1.032,1.010,1.000,1.010,,,
2003,1.169,1.056,1.032,1.010,1.000,,,,
2004,1.169,1.056,1.032,1.010,,,,,
2005,1.169,1.056,1.032,,,,,,
2006,1.169,1.056,,,,,,,
2007,1.169,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
volume_5,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,1.000


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120,120-132
Selected,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,1.000,1.000


,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult,120-Ult
CDF to Ultimate,1.300,1.112,1.053,1.020,1.010,1.010,1.000,1.000,1.000,1.000
Percent Reported,0.769,0.899,0.950,0.980,0.990,0.990,1.000,1.000,1.000,1.000


### Exhibit III, Sheet 5: USPP Auto Increasing Claim Ratios - Paid Claims

We then generate Sheet 5, Parts 1-4 which contain the data triangle and age-to-age factors for USPP paid claims under the increasing claim ratio scenario (p. 118).

In [38]:
_ = dev_exhibit(triangles.loc["Increasing Claim", "Paid Claims"],avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},selected_avg='volume_5',tail=1)

''

,12,24,36,48,60,72,84,96,108,120
1999,"294,000","497,000","588,000","644,000","672,000","686,000","693,000","693,000","700,000","700,000"
2000,"308,700","521,850","617,400","676,200","705,600","720,300","727,650","727,650","735,000",
2001,"324,135","547,943","648,270","710,010","740,880","756,315","764,033","764,033",,
2002,"340,342","575,340","680,684","745,511","777,924","794,131","802,234",,,
2003,"357,359","604,107","714,718","782,786","816,820","833,837",,,,
2004,"428,831","724,928","857,661","939,343","980,184",,,,,
2005,"478,414","808,748","956,828","1,047,955",,,,,,
2006,"531,884","899,137","1,063,768",,,,,,,
2007,"589,505","996,544",,,,,,,,
2008,"651,558",,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1999,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000
2000,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,
2001,1.690,1.183,1.095,1.043,1.021,1.010,1.000,,
2002,1.690,1.183,1.095,1.043,1.021,1.010,,,
2003,1.690,1.183,1.095,1.043,1.021,,,,
2004,1.690,1.183,1.095,1.043,,,,,
2005,1.690,1.183,1.095,,,,,,
2006,1.690,1.183,,,,,,,
2007,1.690,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
volume_5,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120,120-132
Selected,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000,1.000


,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult,120-Ult
CDF to Ultimate,2.378,1.407,1.190,1.086,1.042,1.020,1.010,1.010,1.000,1.000
Percent Reported,0.421,0.711,0.840,0.921,0.960,0.980,0.990,0.990,1.000,1.000


### Exhibit III, Sheet 6: USPP Auto Increasing Case Outstanding Strength - Reported Claims

We then generate Sheet 6, Parts 1-4 which contain the data triangle and age-to-age factors for USPP reported claims under the increasing case outstanding strength scenario (p. 119).

In [39]:
_ = dev_exhibit(triangles.loc["Increasing Case", "Reported Claims"],avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},selected_avg='volume_5',tail=1)

''

,12,24,36,48,60,72,84,96,108,120
1999,"539,000","630,000","665,000","686,000","693,000","693,000","700,000","700,000","700,000","700,000"
2000,"565,950","661,500","698,250","720,300","727,650","727,650","735,000","735,000","735,000",
2001,"594,248","694,575","733,163","756,315","764,033","764,033","771,750","771,750",,
2002,"623,960","729,304","769,821","794,131","802,234","802,234","810,338",,,
2003,"655,158","765,769","808,312","833,837","842,346","842,346",,,,
2004,"687,916","804,057","848,727","878,745","884,463",,,,,
2005,"722,312","844,260","897,355","933,377",,,,,,
2006,"758,427","897,702","962,808",,,,,,,
2007,"818,067","979,922",,,,,,,,
2008,"931,185",,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1999,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,1.000
2000,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,
2001,1.169,1.056,1.032,1.010,1.000,1.010,1.000,,
2002,1.169,1.056,1.032,1.010,1.000,1.010,,,
2003,1.169,1.056,1.032,1.010,1.000,,,,
2004,1.169,1.056,1.035,1.007,,,,,
2005,1.169,1.063,1.040,,,,,,
2006,1.184,1.073,,,,,,,
2007,1.198,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
volume_5,1.178,1.061,1.034,1.009,1.000,1.010,1.000,1.000,1.000


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120,120-132
Selected,1.178,1.061,1.034,1.009,1.000,1.010,1.000,1.000,1.000,1.000


,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult,120-Ult
CDF to Ultimate,1.317,1.118,1.054,1.019,1.010,1.010,1.000,1.000,1.000,1.000
Percent Reported,0.759,0.894,0.949,0.981,0.990,0.990,1.000,1.000,1.000,1.000


### Exhibit III, Sheet 7: USPP Auto Increasing Case Outstanding Strength - Paid Claims

We then generate Sheet 7, Parts 1-4 which contain the data triangle and age-to-age factors for USPP paid claims under the increasing case outstanding strength scenario (p. 120).

In [40]:
_ = dev_exhibit(triangles.loc["Increasing Case", "Paid Claims"],avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},selected_avg='volume_5',tail=1)

''

,12,24,36,48,60,72,84,96,108,120
1999,"294,000","497,000","588,000","644,000","672,000","686,000","693,000","693,000","700,000","700,000"
2000,"308,700","521,850","617,400","676,200","705,600","720,300","727,650","727,650","735,000",
2001,"324,135","547,943","648,270","710,010","740,880","756,315","764,033","764,033",,
2002,"340,342","575,340","680,684","745,511","777,924","794,131","802,234",,,
2003,"357,359","604,107","714,718","782,786","816,820","833,837",,,,
2004,"375,227","634,312","750,454","821,925","857,661",,,,,
2005,"393,988","666,028","787,976","863,022",,,,,,
2006,"413,688","699,329","827,375",,,,,,,
2007,"434,372","734,295",,,,,,,,
2008,"456,090",,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1999,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000
2000,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,
2001,1.690,1.183,1.095,1.043,1.021,1.010,1.000,,
2002,1.690,1.183,1.095,1.043,1.021,1.010,,,
2003,1.690,1.183,1.095,1.043,1.021,,,,
2004,1.690,1.183,1.095,1.043,,,,,
2005,1.690,1.183,1.095,,,,,,
2006,1.690,1.183,,,,,,,
2007,1.690,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
volume_5,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120,120-132
Selected,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000,1.000


,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult,120-Ult
CDF to Ultimate,2.378,1.407,1.190,1.086,1.042,1.020,1.010,1.010,1.000,1.000
Percent Reported,0.421,0.711,0.840,0.921,0.960,0.980,0.990,0.990,1.000,1.000


### Exhibit III, Sheet 8: USPP Auto Increasing Claim Ratios and Increasing Case Outstanding Strength - Reported Claims

We then generate Sheet 8, Parts 1-4 which contain the data triangle and age-to-age factors for USPP reported claims under the increasing claims ratio / increasing case outstanding strength scenario (p. 121).

In [41]:
_ = dev_exhibit(triangles.loc["Increasing Claim Case", "Reported Claims"],avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},selected_avg='volume_5',tail=1)

''

,12,24,36,48,60,72,84,96,108,120
1999,"539,000","630,000","665,000","686,000","693,000","693,000","700,000","700,000","700,000","700,000"
2000,"565,950","661,500","698,250","720,300","727,650","727,650","735,000","735,000","735,000",
2001,"594,248","694,575","733,163","756,315","764,033","764,033","771,750","771,750",,
2002,"623,960","729,304","769,821","794,131","802,234","802,234","810,338",,,
2003,"655,158","765,769","808,312","833,837","842,346","842,346",,,,
2004,"786,189","918,923","969,974","1,004,280","1,010,815",,,,,
2005,"877,093","1,025,173","1,089,645","1,133,386",,,,,,
2006,"975,121","1,154,188","1,237,897",,,,,,,
2007,"1,110,234","1,329,895",,,,,,,,
2008,"1,330,264",,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1999,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,1.000
2000,1.169,1.056,1.032,1.010,1.000,1.010,1.000,1.000,
2001,1.169,1.056,1.032,1.010,1.000,1.010,1.000,,
2002,1.169,1.056,1.032,1.010,1.000,1.010,,,
2003,1.169,1.056,1.032,1.010,1.000,,,,
2004,1.169,1.056,1.035,1.007,,,,,
2005,1.169,1.063,1.040,,,,,,
2006,1.184,1.073,,,,,,,
2007,1.198,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
volume_5,1.179,1.061,1.035,1.009,1.000,1.010,1.000,1.000,1.000


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120,120-132
Selected,1.179,1.061,1.035,1.009,1.000,1.010,1.000,1.000,1.000,1.000


,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult,120-Ult
CDF to Ultimate,1.319,1.119,1.055,1.019,1.010,1.010,1.000,1.000,1.000,1.000
Percent Reported,0.758,0.894,0.948,0.981,0.990,0.990,1.000,1.000,1.000,1.000


### Exhibit III, Sheet 9: USPP Auto Increasing Claim Ratios and Increasing Case Outstanding Strength - Paid Claims

We then generate Sheet 9, Parts 1-4 which contain the data triangle and age-to-age factors for USPP paid claims under the increasing claims ratio / increasing case outstanding strength scenario (p. 122):

In [42]:
_ = dev_exhibit(triangles.loc["Increasing Claim Case", "Paid Claims"],avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},selected_avg='volume_5',tail=1)

''

,12,24,36,48,60,72,84,96,108,120
1999,"294,000","497,000","588,000","644,000","672,000","686,000","693,000","693,000","700,000","700,000"
2000,"308,700","521,850","617,400","676,200","705,600","720,300","727,650","727,650","735,000",
2001,"324,135","547,943","648,270","710,010","740,880","756,315","764,033","764,033",,
2002,"340,342","575,340","680,684","745,511","777,924","794,131","802,234",,,
2003,"357,359","604,107","714,718","782,786","816,820","833,837",,,,
2004,"428,831","724,928","857,661","939,343","980,184",,,,,
2005,"478,414","808,748","956,828","1,047,955",,,,,,
2006,"531,884","899,137","1,063,768",,,,,,,
2007,"589,505","996,544",,,,,,,,
2008,"651,558",,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1999,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000
2000,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,
2001,1.690,1.183,1.095,1.043,1.021,1.010,1.000,,
2002,1.690,1.183,1.095,1.043,1.021,1.010,,,
2003,1.690,1.183,1.095,1.043,1.021,,,,
2004,1.690,1.183,1.095,1.043,,,,,
2005,1.690,1.183,1.095,,,,,,
2006,1.690,1.183,,,,,,,
2007,1.690,,,,,,,,


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
volume_5,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000


,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120,120-132
Selected,1.690,1.183,1.095,1.043,1.021,1.010,1.000,1.010,1.000,1.000


,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult,120-Ult
CDF to Ultimate,2.378,1.407,1.190,1.086,1.042,1.020,1.010,1.010,1.000,1.000
Percent Reported,0.421,0.711,0.840,0.921,0.960,0.980,0.990,0.990,1.000,1.000


In [ ]:
#| hide-input

def Ex10Sht10(scenario, tr):

    def format_col(x, reverse = False):
        if reverse is True:
            return x.to_frame().squeeze()[::-1].reset_index(drop=True)
        return x.to_frame().squeeze().reset_index(drop=True)

    def col_diff(x, y):
        return x.squeeze() - y.squeeze()

    def col_mult(x, y):
        return x.squeeze() * y.squeeze()

    tri = tr.copy()

    dev_input = {
        "average" : "volume",
        "n_periods" : 5
    }

    tail_input = {
        "tail" : 1.0,
        "projection_period" : -1
    }

    dev = cl.Development(**dev_input)
    tail = cl.TailConstant(**tail_input)
    tri = tail.fit_transform(dev.fit_transform(tri))

    actual_scenario = (
    "Steady State"
    if scenario in ["Steady State", "Increasing Case"]
    else "Increasing Claim"
    )

    cdf = format_col(tri.cdf_.loc[actual_scenario]["Reported Claims"], reverse=True)
    ult = format_col(tri.loc[actual_scenario]["Reported Claims"].latest_diagonal)
    
    col1 = pd.DataFrame({
        "Accident Year" : format_col(tr.origin)
        })

    col2 = pd.DataFrame({
        "Age of Accident Year at 12/31/08" : format_col(tr.development, reverse=True)

    })

    col3 = pd.DataFrame({
        "Claims at 12/31/2008 - Reported" : format_col(tri.loc[scenario]["Reported Claims"].latest_diagonal)

    })

    col4 = pd.DataFrame({
        "Claims at 12/31/2008 - Paid" : format_col(tri.loc[scenario]["Paid Claims"].latest_diagonal)

    })

    col5 = pd.DataFrame({
        "Case Outstanding" : col_diff(col3, col4)

    })

    col6 = pd.DataFrame({
        "CDF to Ult. - Reported" : format_col(tri.cdf_.loc[scenario]["Reported Claims"], reverse=True)

    })

    col7 = pd.DataFrame({
        "CDF to Ult. - Paid" : format_col(tri.cdf_.loc[scenario]["Paid Claims"], reverse=True)
    })

    col8 = pd.DataFrame({
        "Projected Ult. Claims Using Dev. Method - Reported" : col_mult(col3, col6)
    })

    col9 = pd.DataFrame({
        "Projected Ult. Claims Using Dev. Method - Paid" : col_mult(col4, col7)
    })

    col10 = pd.DataFrame({
        "Estimated IBNR Using Dev. Method - Reported" : col_diff(col8, col3)
    })

    col11 = pd.DataFrame({
        "Estimated IBNR Using Dev. Method - Paid" : col_diff(col9, col3)
    })

    col12 = pd.DataFrame({
        "Actual IBNR" : col_diff(col_mult(ult, cdf), col3)
    })

    col13 = pd.DataFrame({
        "Difference from Actual IBNR - Reported" : col_diff(col12, col10)
    })

    col14 = pd.DataFrame({
        "Difference from Actual IBNR - Paid" : col_diff(col12, col11)
    })


    cols = [col1, col2, col3, col4, col5, col6, col7, col8, col9, col10, col11, col12, col13, col14]

    results = pd.concat(cols, axis=1)

    

    display(
        results
    )



### Exhibit III, Sheets 10 and 11: USPP Auto Development of Unpaid Claim Estimate

We then generate Sheets 10 and 11 which illustrate the differences between the actual IBNR and the IBNR obtained through the development method for each scenario (pp. 123-124).

In [ ]:
scenarios = ["Steady State","Increasing Claim","Increasing Case","Increasing Claim Case"]

for scenario in scenarios:
    Ex10Sht10(scenario, triangles)

,Accident Year,Age of Accident Year at 12/31/08,Claims at 12/31/2008 - Reported,Claims at 12/31/2008 - Paid,Case Outstanding,CDF to Ult. - Reported,CDF to Ult. - Paid,Projected Ult. Claims Using Dev. Method - Reported,Projected Ult. Claims Using Dev. Method - Paid,Estimated IBNR Using Dev. Method - Reported,Estimated IBNR Using Dev. Method - Paid,Actual IBNR,Difference from Actual IBNR - Reported,Difference from Actual IBNR - Paid
0,1999,120,"700,000","700,000",0,1,1,"700,000","700,000",0,0,0,0,0
1,2000,108,"735,000","735,000",0,1,1,"735,000","735,000",0,0,0,0,0
2,2001,96,"771,750","764,033","7,717",1,1,"771,750","771,751",0,1,0,0,-1
3,2002,84,"810,338","802,234","8,104",1,1,"810,338","810,337",0,-1,0,0,1
4,2003,72,"842,346","833,837","8,509",1,1,"850,855","850,854","8,509","8,508","8,509",0,0
5,2004,60,"884,463","857,661","26,802",1,1,"893,397","893,397","8,934","8,934","8,934",0,0
6,2005,48,"919,306","863,022","56,284",1,1,"938,068","938,067","18,762","18,761","18,762",0,0
7,2006,36,"935,722","827,375","108,347",1,1,"984,970","984,970","49,248","49,248","49,248",0,0
8,2007,24,"930,797","734,295","196,502",1,1,"1,034,219","1,034,218","103,422","103,421","103,422",0,1
9,2008,12,"836,166","456,090","380,076",1,2,"1,085,930","1,085,928","249,764","249,762","249,764",0,2


,Accident Year,Age of Accident Year at 12/31/08,Claims at 12/31/2008 - Reported,Claims at 12/31/2008 - Paid,Case Outstanding,CDF to Ult. - Reported,CDF to Ult. - Paid,Projected Ult. Claims Using Dev. Method - Reported,Projected Ult. Claims Using Dev. Method - Paid,Estimated IBNR Using Dev. Method - Reported,Estimated IBNR Using Dev. Method - Paid,Actual IBNR,Difference from Actual IBNR - Reported,Difference from Actual IBNR - Paid
0,1999,120,"700,000","700,000",0,1,1,"700,000","700,000",0,0,0,0,0
1,2000,108,"735,000","735,000",0,1,1,"735,000","735,000",0,0,0,0,0
2,2001,96,"771,750","764,033","7,717",1,1,"771,750","771,751",0,1,0,0,-1
3,2002,84,"810,338","802,234","8,104",1,1,"810,338","810,337",0,-1,0,0,1
4,2003,72,"842,346","833,837","8,509",1,1,"850,855","850,854","8,509","8,508","8,509",0,0
5,2004,60,"1,010,815","980,184","30,631",1,1,"1,021,025","1,021,025","10,210","10,210","10,210",0,0
6,2005,48,"1,116,300","1,047,955","68,345",1,1,"1,139,082","1,139,081","22,782","22,781","22,782",0,0
7,2006,36,"1,203,071","1,063,768","139,303",1,1,"1,266,391","1,266,390","63,320","63,319","63,320",0,0
8,2007,24,"1,263,224","996,544","266,680",1,1,"1,403,582","1,403,583","140,358","140,359","140,358",0,-0
9,2008,12,"1,194,523","651,558","542,965",1,2,"1,551,328","1,551,328","356,805","356,805","356,805",0,0


,Accident Year,Age of Accident Year at 12/31/08,Claims at 12/31/2008 - Reported,Claims at 12/31/2008 - Paid,Case Outstanding,CDF to Ult. - Reported,CDF to Ult. - Paid,Projected Ult. Claims Using Dev. Method - Reported,Projected Ult. Claims Using Dev. Method - Paid,Estimated IBNR Using Dev. Method - Reported,Estimated IBNR Using Dev. Method - Paid,Actual IBNR,Difference from Actual IBNR - Reported,Difference from Actual IBNR - Paid
0,1999,120,"700,000","700,000",0,1,1,"700,000","700,000",0,0,0,0,0
1,2000,108,"735,000","735,000",0,1,1,"735,000","735,000",0,0,0,0,0
2,2001,96,"771,750","764,033","7,717",1,1,"771,750","771,751",0,1,0,0,-1
3,2002,84,"810,338","802,234","8,104",1,1,"810,338","810,337",0,-1,0,0,1
4,2003,72,"842,346","833,837","8,509",1,1,"850,855","850,854","8,509","8,508","8,509",0,0
5,2004,60,"884,463","857,661","26,802",1,1,"893,397","893,397","8,934","8,934","8,934",0,0
6,2005,48,"933,377","863,022","70,355",1,1,"951,657","938,067","18,280","4,690","4,691","-13,589",0
7,2006,36,"962,808","827,375","135,433",1,1,"1,015,301","984,970","52,493","22,162","22,162","-30,331",0
8,2007,24,"979,922","734,295","245,627",1,1,"1,096,235","1,034,218","116,313","54,296","54,297","-62,016",1
9,2008,12,"931,185","456,090","475,095",1,2,"1,227,589","1,085,928","296,404","154,743","154,745","-141,659",2


,Accident Year,Age of Accident Year at 12/31/08,Claims at 12/31/2008 - Reported,Claims at 12/31/2008 - Paid,Case Outstanding,CDF to Ult. - Reported,CDF to Ult. - Paid,Projected Ult. Claims Using Dev. Method - Reported,Projected Ult. Claims Using Dev. Method - Paid,Estimated IBNR Using Dev. Method - Reported,Estimated IBNR Using Dev. Method - Paid,Actual IBNR,Difference from Actual IBNR - Reported,Difference from Actual IBNR - Paid
0,1999,120,"700,000","700,000",0,1,1,"700,000","700,000",0,0,0,0,0
1,2000,108,"735,000","735,000",0,1,1,"735,000","735,000",0,0,0,0,0
2,2001,96,"771,750","764,033","7,717",1,1,"771,750","771,751",0,1,0,0,-1
3,2002,84,"810,338","802,234","8,104",1,1,"810,338","810,337",0,-1,0,0,1
4,2003,72,"842,346","833,837","8,509",1,1,"850,855","850,854","8,509","8,508","8,509",0,0
5,2004,60,"1,010,815","980,184","30,631",1,1,"1,021,025","1,021,025","10,210","10,210","10,210",0,0
6,2005,48,"1,133,386","1,047,955","85,431",1,1,"1,155,482","1,139,081","22,096","5,695","5,696","-16,400",0
7,2006,36,"1,237,897","1,063,768","174,129",1,1,"1,305,639","1,266,390","67,742","28,493","28,494","-39,249",0
8,2007,24,"1,329,895","996,544","333,351",1,1,"1,488,875","1,403,583","158,980","73,688","73,687","-85,293",-0
9,2008,12,"1,330,264","651,558","678,706",1,2,"1,756,504","1,551,328","426,240","221,064","221,064","-205,176",0


## Exhibit IV — Impact of Changing Product Mix Example

In order to illustrate the assumptions and corresponding limitations of the Development method, Exhibit III applies the method to the **U.S. Private Passenger Industry Auto** (USPP Auto) example under four different scenarios:

1. stable claim ratios and no change in case outstanding strength (steady state);
1. increasing claim ratios but no change in case outstanding strength;
1. stable claim ratios but increasing case outstanding strength; and
1. increasing claim ratios and increasing case outstanding strength.

The key assumptions of this study are as follows:

- the earned premium for the first year (1999) is assumed to be $1 million with a 5% annual premium trend;
- (discuss assumed claim ratios)

Since the USPP Auto dataset is contained in the `chainladder` package, we simply load it into a triangle as follows:

,12,24,36,48,60,72,84,96,108,120
1999,"1,011,000","1,254,000","1,377,000","1,454,000","1,477,000","1,493,000","1,500,000","1,500,000","1,500,000","1,500,000"
2000,"1,061,550","1,316,700","1,445,850","1,526,700","1,550,850","1,567,650","1,575,000","1,575,000","1,575,000",
2001,"1,114,628","1,382,535","1,518,143","1,603,035","1,628,393","1,646,033","1,653,750","1,653,750",,
2002,"1,170,359","1,451,662","1,594,050","1,683,187","1,709,812","1,728,334","1,736,438",,,
2003,"1,228,877","1,524,245","1,673,752","1,767,346","1,795,303","1,814,751",,,,
2004,"1,290,321","1,600,457","1,757,440","1,855,713","1,885,068",,,,,
2005,"1,354,837","1,680,480","1,845,312","1,948,499",,,,,,
2006,"1,422,579","1,764,504","1,937,577",,,,,,,
2007,"1,493,707",,,,,,,,,
2008,"3,421,122",,,,,,,,,
